# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 8 · Parallel Swarm — fault-tolerant merge

**Phase B · Orchestration** &nbsp;|&nbsp; **Goal:** parallelise the per-unit work and merge the best results.

---

### What Lab 8 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | Per-department metric sources, one per unit. |
| **Outputs** | One merged institution view, with coverage and conflicts resolved. |
| **Agents / nodes** | A swarm of 8 department agents + a merge step. |
| **Deliverable** | Parallel per-unit execution with retry, partial results and conflict resolution. |
| **Acceptance** | Work runs in parallel and results are **merged**, with conflicts resolved rather than summed. |

### What this adds over Lab 7

Lab 7 already ran four domain agents in one parallel step. Lab 8 is about everything that happens when parallel work **goes wrong**:

| Failure mode | What the swarm does |
|---|---|
| an agent is slow | timeout |
| an agent fails | retry with backoff — **CHEM** recovers on attempt 2 |
| an agent stays failed | **CIVIL** is given up on; the merge continues at 87.5% coverage |
| two agents disagree | **CSE and ECE both claim the same three joint papers** — counted once, not twice |

Here the unit is the **department**: eight agents, one each.

### The finding this lab produces

Chapter 15's Research Agent guardrail says *"never double-count a cross-department collaboration."* This notebook enforces it in the merge — and in doing so uncovers something about the data Labs 1–7 were using:

| | publications | KPI-R1 |
|---|---|---|
| flat department-reports export | **185** | 1.1859 per faculty |
| merged by publication **identity** | **182** | 1.1667 per faculty |

The export was never wrong about any single department — each correctly reported its own joint papers. The double-count only becomes visible when units are merged by **identity** rather than by **count**, which is exactly what a swarm-plus-merge does and a flat sum cannot.

### How to run

Top to bottom, **no API key and no network**. Agents sleep 60 ms each to model a real departmental system, so the parallel speed-up is measurable. Cell 7 executes the rubric as assertions.

---

### Cell 1 · The accumulated stack

`edss_core.py` (Labs 1–2) and `edss_stack.py` (Labs 3–5), written to disk so this notebook runs standalone.

*(Collapse both — prior work.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Overwriting edss_core.py


In [2]:
%%writefile edss_stack.py
"""edss_stack — the capabilities Labs 3, 4 and 5 added, in one importable layer.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

    Lab 3  Skill            reusable, schema-typed capability packages
    Lab 4  RunState/Archive short-term run state vs the long-term baseline,
                            with tag and TF-IDF-similarity retrieval
    Lab 5  MCPServer/Client one typed, scoped connector over file + SQLite + API

Labs 6-8 bind these together rather than restating them: Lab 6 puts a runtime
around them, Lab 7 wires them into a node graph, Lab 8 runs them in parallel.

Written to disk by the first code cells of Labs 6-8 so each notebook is
standalone.
"""

import csv
import io
import json
import math
import re
import sqlite3
import textwrap
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ACCESS_REGISTER,
                       DEPARTMENT_REPORTS_CSV, NUMERIC_COLUMNS, BENCHMARK_API_DB,
                       ToolRouter, Connector, ToolError, ToolContractError,
                       compute_kpi, compare_to_target, validate_kpi_result,
                       _typed_rows)

_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


# ==========================================================================
# LAB 3 — the Skill framework
# ==========================================================================

class SkillContractError(Exception):
    """Raised when a skill's input or output violates its declared schema."""


@dataclass
class Skill:
    """A reusable capability package.

    name          stable identifier
    description   what it does, in one line
    instructions  the procedure it follows — the part a human or a model reads
    constraints   what it must never do
    input_schema  typed contract, validated BEFORE the skill runs
    output_schema typed contract, validated AFTER it runs
    tools         the tool names it is permitted to call
    """
    name: str
    description: str
    instructions: str
    constraints: List[str]
    input_schema: Dict[str, Any]
    output_schema: Dict[str, Any]
    tools: List[str]
    _fn: Optional[Callable] = None
    invocations: List[Dict[str, Any]] = field(default_factory=list)

    # ---- schema validation, both directions ----------------------------
    def _validate(self, payload: Dict[str, Any], schema: Dict[str, Any],
                  where: str) -> None:
        props, required = schema.get("properties", {}), schema.get("required", [])
        for name in required:
            if name not in payload:
                raise SkillContractError(
                    f"{self.name} {where}: missing required field '{name}'")
        for name, value in payload.items():
            if name not in props:
                raise SkillContractError(
                    f"{self.name} {where}: unexpected field '{name}'; "
                    f"accepts {sorted(props)}")
            expected = _JSON_TYPES.get(props[name]["type"])
            if expected and not isinstance(value, expected):
                raise SkillContractError(
                    f"{self.name} {where}.{name}: expected {props[name]['type']}, "
                    f"got {type(value).__name__}")
            if props[name].get("enum") and value not in props[name]["enum"]:
                raise SkillContractError(
                    f"{self.name} {where}.{name}: {value!r} not in "
                    f"{props[name]['enum']}")

    def run(self, router: "ToolRouter", **inputs) -> Dict[str, Any]:
        self._validate(inputs, self.input_schema, "input")
        before = len(router.trace)
        result = self._fn(router, **inputs)
        self._validate(result, self.output_schema, "output")
        used = [e["tool"] for e in router.trace[before:]]
        stray = [t for t in used if t not in self.tools]
        if stray:
            raise SkillContractError(
                f"{self.name} called tools outside its declared surface: {stray}")
        self.invocations.append({"inputs": inputs, "tools_used": used,
                                 "output_keys": sorted(result)})
        return result

    def card(self) -> str:
        return (f"SKILL  {self.name}\n"
                f"  {self.description}\n"
                f"  inputs   : {list(self.input_schema.get('properties', {}))}"
                f"  (required: {self.input_schema.get('required', [])})\n"
                f"  outputs  : {list(self.output_schema.get('properties', {}))}\n"
                f"  tools    : {self.tools}\n"
                f"  constraints:\n"
                + "\n".join(f"    - {c}" for c in self.constraints))


SKILL_REGISTRY: Dict[str, Skill] = {}


def register(skill: Skill) -> Skill:
    SKILL_REGISTRY[skill.name] = skill
    return skill


# ==========================================================================
# LAB 3 — skill: review_domain
# ==========================================================================

REVIEW_DOMAIN = Skill(
    name="review_domain",
    description="Review one domain's approved KPIs against their targets, with "
                "per-department evidence.",
    instructions=(
        "1. Read the department rows for the requested period through the connector.\n"
        "2. If a department filter is supplied, narrow to it and refuse an empty subset.\n"
        "3. For every approved KPI belonging to the domain: read its definition, compute\n"
        "   it with the registered formula, and compare it to its approved target.\n"
        "4. Return the KPI set with variance, status, source system and figure status.\n"
        "5. Report which KPIs are off target; never recommend an action — that is the\n"
        "   Report Generation Agent's job, behind the human gate."),
    constraints=[
        "Use only approved KPI definitions; never invent a formula.",
        "Never average department percentages — aggregate raw counts.",
        "Label any figure that is not final; never present it as an actual.",
        "Emit no recommendation and take no action.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string", "enum": DOMAINS},
            "period": {"type": "string"},
            "category": {"type": "string", "enum": ["circuit", "core"]},
        },
        "required": ["domain", "period"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string"},
            "period": {"type": "string"},
            "scope": {"type": "string"},
            "kpis": {"type": "array"},
            "off_target": {"type": "array"},
            "headline": {"type": "string"},
            "provisional": {"type": "boolean"},
        },
        "required": ["domain", "period", "scope", "kpis", "off_target", "headline",
                     "provisional"],
    },
    tools=["read_department_reports", "filter_rows", "read_kpi_definitions",
           "compute_kpi", "compare_to_target"],
)


def _review_domain(router, domain, period, category=None):
    connector = Connector(router)
    rows = connector.read("department reports",
                          reason=f"review_domain({domain}): per-department metrics",
                          period=period)["rows"]
    scope = "all departments"
    if category:
        rows = router.call("filter_rows",
                           reason=f"review_domain({domain}): narrow to {category} branches",
                           rows=rows, field="category", value=category)["rows"]
        scope = f"{category} departments"

    kpi_ids = [k for k, d in KPI_DEFINITIONS.items() if d["domain"] == domain]
    kpis, provisional = [], False
    for kpi_id in kpi_ids:
        connector.read("institutional KPIs & dashboards",
                       reason=f"{kpi_id}: approved formula and target",
                       kpi_ids=[kpi_id])
        computed = router.call("compute_kpi", reason=f"{kpi_id}: apply the formula",
                               kpi_id=kpi_id, rows=rows)
        compared = router.call("compare_to_target", reason=f"{kpi_id}: variance vs target",
                               kpi_id=kpi_id, value=computed["value"])
        gate = validate_kpi_result(computed, compared)
        if not gate.valid:
            raise ToolError(f"validation gate failed for {kpi_id}: {gate.failing}")
        provisional = provisional or computed["figure_status"] != "final"
        kpis.append({
            "kpi_id": kpi_id, "name": computed["name"], "value": computed["value"],
            "unit": computed["unit"], "target": compared["target"],
            "target_source": compared["target_source"], "variance": compared["variance"],
            "status": compared["status"], "formula": computed["formula"],
            "source_system": computed["source_system"],
            "figure_status": computed["figure_status"],
            "rounding_warning": compared["rounding_warning"],
            "evidence": computed["evidence"],
        })

    off = [k["kpi_id"] for k in kpis if k["status"] != "on target"]
    headline = (f"{domain}: {len(off)} of {len(kpis)} KPIs off target across {scope} "
                f"for {period}." + (" Figures include a provisional source."
                                    if provisional else ""))
    return {"domain": domain, "period": period, "scope": scope, "kpis": kpis,
            "off_target": off, "headline": headline, "provisional": provisional}


REVIEW_DOMAIN._fn = _review_domain
register(REVIEW_DOMAIN)


# ==========================================================================
# LAB 3 — skill: format_document
# ==========================================================================

FORMAT_DOCUMENT = Skill(
    name="format_document",
    description="Render a titled, sectioned payload as a leadership-ready document "
                "in a chosen template.",
    instructions=(
        "1. Emit the title, then the metadata block as key: value lines.\n"
        "2. Emit each section in order: a heading, then its bullets, then its table\n"
        "   if it has one.\n"
        "3. In 'brief' template, number the sections; in 'memo' template, do not.\n"
        "4. Append the source line verbatim if one is supplied.\n"
        "5. Reproduce every value exactly as given — never reformat a number."),
    constraints=[
        "Never compute, round or reword a value; this skill only lays out.",
        "Never drop a section or a caveat that was passed in.",
        "Stay domain-agnostic: no KPI, department or worklet logic here.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "meta": {"type": "object"},
            "sections": {"type": "array"},
            "template": {"type": "string", "enum": ["brief", "memo"]},
            "source_line": {"type": "string"},
        },
        "required": ["title", "sections"],
    },
    output_schema={
        "type": "object",
        "properties": {"document": {"type": "string"},
                       "template": {"type": "string"},
                       "section_count": {"type": "integer"},
                       "line_count": {"type": "integer"}},
        "required": ["document", "template", "section_count", "line_count"],
    },
    tools=[],          # a pure formatting skill: it calls no tool at all
)


def _format_document(router, title, sections, meta=None, template="brief",
                     source_line=None):
    out = [title, "=" * len(title)]
    for k, v in (meta or {}).items():
        out.append(f"{k}: {v}")
    out.append("")
    for i, sec in enumerate(sections, 1):
        heading = sec.get("heading", "")
        out.append(f"{i}. {heading}" if template == "brief" else heading.upper())
        out.append("-" * 70)
        for b in sec.get("bullets", []):
            out.append(textwrap.fill(f"  - {b}", 88, subsequent_indent="    "))
        table = sec.get("table")
        if table:
            widths = [max(len(str(r[c])) for r in [dict(zip(table["columns"],
                                                            table["columns"]))] + table["rows"])
                      for c in table["columns"]]
            out.append("  " + "  ".join(str(c).ljust(w) for c, w in
                                        zip(table["columns"], widths)))
            for row in table["rows"]:
                out.append("  " + "  ".join(str(row[c]).ljust(w) for c, w in
                                            zip(table["columns"], widths)))
        out.append("")
    if source_line:
        out.append(source_line)
    document = "\n".join(out)
    return {"document": document, "template": template,
            "section_count": len(sections), "line_count": len(document.splitlines())}


FORMAT_DOCUMENT._fn = _format_document
register(FORMAT_DOCUMENT)


# ==========================================================================
# LAB 4 — the memory split
# ==========================================================================

class ArchiveImmutableError(Exception):
    """Raised on any attempt to edit a record that is already archived."""


@dataclass
class RunState:
    """Short-term working memory. One run. Thrown away at the end."""
    run_id: str
    period: str
    draft: Dict[str, Any] = field(default_factory=dict)
    scratch: Dict[str, Any] = field(default_factory=dict)
    retrieved: List[Dict[str, Any]] = field(default_factory=list)

    def note(self, key: str, value: Any) -> None:
        self.scratch[key] = value

    def summary(self) -> str:
        return (f"RunState({self.run_id}) draft_keys={sorted(self.draft)} "
                f"scratch_keys={sorted(self.scratch)} retrieved={len(self.retrieved)}")


class Archive:
    """Long-term store. Append-only; every write is versioned and timestamped."""

    def __init__(self):
        self._records: Dict[str, Dict[str, Any]] = {}
        self.writes: List[Dict[str, Any]] = []

    # ---- writing -------------------------------------------------------
    def put(self, record_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
        if record_id in self._records:
            raise ArchiveImmutableError(
                f"'{record_id}' is already archived; the archive is append-only. "
                f"Write a new version id instead of editing the baseline.")
        stored = {**record, "record_id": record_id,
                  "archived_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self._records[record_id] = stored
        self.writes.append({"record_id": record_id, "tags": record.get("tags", []),
                            "at": stored["archived_at"]})
        return stored

    # ---- reading -------------------------------------------------------
    def get(self, record_id: str) -> Dict[str, Any]:
        # A COPY, so a caller mutating what it read cannot corrupt the baseline.
        return json.loads(json.dumps(self._records[record_id]))

    def all(self) -> List[Dict[str, Any]]:
        return [self.get(rid) for rid in self._records]

    def __len__(self):
        return len(self._records)

    def __contains__(self, record_id):
        return record_id in self._records


# ==========================================================================
# LAB 4 — what is in the archive
# ==========================================================================

archive = Archive()

PRIOR_BRIEFS = [
    {"record_id": "BRIEF-2023-24-Q4",
     "type": "decision_brief", "period": "2023-24",
     "title": "Placement recovery after the 2023 hiring slowdown",
     "tags": ["placement", "core-branches", "recruiter-outreach", "below-target"],
     "summary": "Placement closed at 71.80% against an 80% target. Core branches drove "
                "the gap; circuit branches held. The Council approved a recruiter "
                "outreach drive and deferred the package target.",
     "decisions": ["Approved INR 12 lakh recruiter outreach budget",
                   "Deferred the average-package target by one year"],
     "outcome": "Placement rose to 73.50% the following year — short of target but "
                "the trend reversed. The wider recruiter pool brought in lower-paying "
                "offers, and average package began to slide."},
    {"record_id": "BRIEF-2024-25-Q2",
     "type": "decision_brief", "period": "2024-25",
     "title": "Research output against the NIRF submission",
     "tags": ["research", "publications", "citations", "nirf", "below-target"],
     "summary": "Publications per faculty stood at 1.05 against a 1.50 target and "
                "citations per faculty at 6.40. The Council approved a seed-grant "
                "scheme and asked for a per-department breakdown each quarter.",
     "decisions": ["Approved a seed-grant scheme for research-active faculty",
                   "Made department-level drill-down mandatory for research KPIs"],
     "outcome": "Publications rose to 1.12; the reporting change stuck and is now "
                "standing practice."},
    {"record_id": "BRIEF-2024-25-Q4",
     "type": "decision_brief", "period": "2024-25",
     "title": "Budget utilisation and the unreconciled Q4 position",
     "tags": ["finance", "budget-utilisation", "provisional", "reconciliation"],
     "summary": "Utilisation was reported at 88.10% but Q4 was unreconciled at the time "
                "of review. The Council declined to act on a provisional figure and "
                "deferred the decision to the post-audit cycle.",
     "decisions": ["Deferred all budget reallocation until the books closed",
                   "Required provisional figures to be labelled on the face of the brief"],
     "outcome": "Final utilisation came in at 86.90% — 1.2 points below the provisional "
                "figure, vindicating the deferral."},
    {"record_id": "BRIEF-2024-25-Q3",
     "type": "decision_brief", "period": "2024-25",
     "title": "Pass percentage and the rounding dispute",
     "tags": ["academic", "pass-percentage", "rounding", "at-risk"],
     "summary": "Pass percentage was circulated as '92%' when the unrounded figure was "
                "91.60%, which is below the 92.00% target. The Council directed that "
                "KPI figures be reported unrounded.",
     "decisions": ["Directed that all KPI figures be reported unrounded",
                   "Ordered remedial coaching in the two weakest departments"],
     "outcome": "The reporting rule is now enforced by the KPI engine itself."},
]

PRIOR_KPI_SNAPSHOTS = [
    {"record_id": "KPI-SNAPSHOT-2023-24", "type": "kpi_snapshot", "period": "2023-24",
     "title": "Institution KPI snapshot 2023-24",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 90.40, "KPI-A2": 2.38, "KPI-R1": 0.98, "KPI-R2": 5.90,
                "KPI-R3": 198.0, "KPI-P1": 71.80, "KPI-P2": 7.10, "KPI-P3": 64.20,
                "KPI-F1": 84.30}},
    {"record_id": "KPI-SNAPSHOT-2024-25", "type": "kpi_snapshot", "period": "2024-25",
     "title": "Institution KPI snapshot 2024-25",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 91.20, "KPI-A2": 2.44, "KPI-R1": 1.12, "KPI-R2": 6.40,
                "KPI-R3": 221.5, "KPI-P1": 73.50, "KPI-P2": 7.95, "KPI-P3": 69.10,
                "KPI-F1": 86.90}},
]

for rec in PRIOR_BRIEFS + PRIOR_KPI_SNAPSHOTS:
    archive.put(rec["record_id"], {k: v for k, v in rec.items() if k != "record_id"})


# ==========================================================================
# LAB 4 — retrieval by tag
# ==========================================================================

def retrieve_by_tags(store: Archive, tags: List[str], mode: str = "any",
                     record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """mode='any' -> union; mode='all' -> records carrying every tag."""
    if mode not in ("any", "all"):
        raise ValueError("mode must be 'any' or 'all'")
    wanted = {t.lower() for t in tags}
    hits = []
    for rec in store.all():
        if record_type and rec.get("type") != record_type:
            continue
        have = {t.lower() for t in rec.get("tags", [])}
        overlap = wanted & have
        if (mode == "any" and overlap) or (mode == "all" and wanted <= have):
            hits.append({**rec, "matched_tags": sorted(overlap),
                         "tag_score": len(overlap) / len(wanted)})
    return sorted(hits, key=lambda r: -r["tag_score"])


# ==========================================================================
# LAB 4 — retrieval by similarity
# ==========================================================================

STOPWORDS = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "was",
             "were", "is", "are", "be", "been", "by", "with", "that", "this", "it",
             "as", "from", "but", "not", "no", "we", "our", "its", "their", "has",
             "had", "have", "than", "then", "so", "which", "when", "all", "one"}


def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9\-]+", text.lower())
            if t not in STOPWORDS and len(t) > 2]


class SimilarityIndex:
    """TF-IDF vectors + cosine similarity over the archive's text."""

    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records
        self.docs = [self._text(r) for r in records]
        self.tokens = [tokenize(d) for d in self.docs]
        n = len(self.tokens)
        df = Counter(t for toks in self.tokens for t in set(toks))
        # Smoothed IDF, so a term in every document still carries a little weight.
        self.idf = {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}
        self.vectors = [self._vector(toks) for toks in self.tokens]

    @staticmethod
    def _text(rec: Dict[str, Any]) -> str:
        parts = [rec.get("title", ""), rec.get("summary", ""),
                 " ".join(rec.get("tags", [])), " ".join(rec.get("decisions", [])),
                 rec.get("outcome", "")]
        return " ".join(p for p in parts if p)

    def _vector(self, toks: List[str]) -> Dict[str, float]:
        if not toks:
            return {}
        tf = Counter(toks)
        vec = {t: (c / len(toks)) * self.idf.get(t, 1.0) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    @staticmethod
    def _cosine(a: Dict[str, float], b: Dict[str, float]) -> float:
        small, large = (a, b) if len(a) < len(b) else (b, a)
        return sum(v * large.get(t, 0.0) for t, v in small.items())

    def query(self, text: str, k: int = 3,
              record_type: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = self._vector(tokenize(text))
        scored = []
        for rec, vec in zip(self.records, self.vectors):
            if record_type and rec.get("type") != record_type:
                continue
            scored.append({**rec, "similarity": round(self._cosine(qv, vec), 4)})
        return sorted(scored, key=lambda r: -r["similarity"])[:k]


index = SimilarityIndex(archive.all())


# ==========================================================================
# LAB 4 — hybrid retrieval
# ==========================================================================

def retrieve(store: Archive, idx: SimilarityIndex, text: str,
             tags: Optional[List[str]] = None, k: int = 3,
             record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """Tag filter first (precision), then similarity rank (relevance)."""
    pool = (retrieve_by_tags(store, tags, "any", record_type) if tags
            else [r for r in store.all()
                  if not record_type or r.get("type") == record_type])
    if not pool:
        return []
    sub = SimilarityIndex(pool)
    hits = sub.query(text, k=k, record_type=record_type)
    by_id = {h["record_id"]: h for h in hits}
    for p in pool:
        if p["record_id"] in by_id and "tag_score" in p:
            by_id[p["record_id"]]["tag_score"] = p["tag_score"]
    return hits


# ==========================================================================
# LAB 5 — the MCP protocol
# ==========================================================================

# ---- error codes: one vocabulary for every backend ------------------------
E_UNKNOWN_RESOURCE = "UNKNOWN_RESOURCE"
E_UNAUTHORIZED     = "UNAUTHORIZED"
E_FORBIDDEN_SCOPE  = "FORBIDDEN_SCOPE"
E_BAD_REQUEST      = "BAD_REQUEST"
E_NOT_FOUND        = "NOT_FOUND"
E_BACKEND_FAILURE  = "BACKEND_FAILURE"


class MCPError(Exception):
    def __init__(self, code: str, message: str):
        super().__init__(f"{code}: {message}")
        self.code, self.message = code, message


@dataclass
class Resource:
    """One advertised capability of the server."""
    name: str
    description: str
    params: Dict[str, Any]          # typed parameter schema
    returns: str
    scope: str                      # the scope a token must hold
    backend: str                    # which backend currently serves it
    handler: Callable = None

    def manifest(self) -> Dict[str, Any]:
        return {"name": self.name, "description": self.description,
                "params": self.params, "returns": self.returns,
                "required_scope": self.scope, "served_by": self.backend}


class MCPServer:
    """One server, many backends. Agents never see which is which."""

    def __init__(self, name: str):
        self.name = name
        self.resources: Dict[str, Resource] = {}
        self.tokens: Dict[str, Dict[str, Any]] = {}
        self.log: List[Dict[str, Any]] = []

    # ---- registration & auth -------------------------------------------
    def expose(self, resource: Resource) -> None:
        self.resources[resource.name] = resource

    def issue_token(self, token: str, holder: str, scopes: List[str]) -> None:
        self.tokens[token] = {"holder": holder, "scopes": set(scopes)}

    def describe(self) -> Dict[str, Any]:
        """Capability discovery — what a client may call, without calling it."""
        return {"server": self.name,
                "resources": [r.manifest() for r in self.resources.values()]}

    # ---- the single entry point ----------------------------------------
    def call(self, token: str, resource: str, **params) -> Dict[str, Any]:
        started = time.perf_counter()
        entry = {"seq": len(self.log) + 1, "resource": resource,
                 "holder": self.tokens.get(token, {}).get("holder", "<unknown>"),
                 "params": {k: (f"<{len(v)} rows>" if isinstance(v, list) else v)
                            for k, v in params.items()},
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}

        def finish(ok, code=None, note=""):
            entry.update({"ok": ok, "error_code": code, "note": note,
                          "ms": round((time.perf_counter() - started) * 1000, 2)})
            self.log.append(entry)

        if token not in self.tokens:
            finish(False, E_UNAUTHORIZED)
            raise MCPError(E_UNAUTHORIZED, "token not recognised by this server")
        if resource not in self.resources:
            finish(False, E_UNKNOWN_RESOURCE)
            raise MCPError(E_UNKNOWN_RESOURCE,
                           f"'{resource}' is not exposed; call describe() to see "
                           f"{sorted(self.resources)}")
        res = self.resources[resource]
        if res.scope not in self.tokens[token]["scopes"]:
            finish(False, E_FORBIDDEN_SCOPE)
            raise MCPError(E_FORBIDDEN_SCOPE,
                           f"'{resource}' needs scope '{res.scope}'; this token holds "
                           f"{sorted(self.tokens[token]['scopes'])}")
        for p, spec in res.params.items():
            if spec.get("required") and p not in params:
                finish(False, E_BAD_REQUEST)
                raise MCPError(E_BAD_REQUEST, f"{resource}: missing parameter '{p}'")
        try:
            data = res.handler(**params)
        except MCPError as exc:
            finish(False, exc.code, exc.message)
            raise
        except Exception as exc:
            finish(False, E_BACKEND_FAILURE, str(exc))
            raise MCPError(E_BACKEND_FAILURE, f"{res.backend} failed: {exc}")
        entry["served_by"] = res.backend
        finish(True)
        # The envelope is identical whatever the backend was.
        return {"ok": True, "resource": resource, "served_by": res.backend, "data": data,
                "meta": {"server": self.name, "at": entry["at"]}}


class MCPClient:
    """What an agent holds. It knows a token and resource names — nothing else."""

    def __init__(self, server: MCPServer, token: str):
        self._server, self._token = server, token

    def describe(self) -> Dict[str, Any]:
        return self._server.describe()

    def read(self, resource: str, **params) -> Any:
        return self._server.call(self._token, resource, **params)["data"]


# ==========================================================================
# LAB 5 — file, SQLite and API backends
# ==========================================================================

WORKDIR = Path("lab5_sources")
WORKDIR.mkdir(exist_ok=True)

# ---- backend 1: a FILE ----------------------------------------------------
CSV_PATH = WORKDIR / "department_reports_2025_26.csv"
CSV_PATH.write_text(DEPARTMENT_REPORTS_CSV)


def _cast(row: Dict[str, str]) -> Dict[str, Any]:
    out = dict(row)
    for col, caster in NUMERIC_COLUMNS.items():
        if col in out:
            out[col] = caster(out[col])
    return out


class FileBackend:
    name = "file:csv"

    def __init__(self, path: Path):
        self.path = path

    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        if period != REPORTING_PERIOD:
            raise MCPError(E_NOT_FOUND, f"no export on disk for '{period}'")
        with self.path.open() as fh:
            rows = [_cast(r) for r in csv.DictReader(fh)]
        if category:
            rows = [r for r in rows if r["category"] == category]
            if not rows:
                raise MCPError(E_NOT_FOUND, f"no departments in category '{category}'")
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports export (CSV on disk)"}


# ---- backend 2: SQLite ----------------------------------------------------
DB_PATH = WORKDIR / "institution.db"
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
CREATE TABLE kpi_definition (
    kpi_id TEXT PRIMARY KEY, name TEXT, domain TEXT, unit TEXT, formula TEXT,
    target REAL, target_source TEXT, source_system TEXT, figure_status TEXT, caveat TEXT);
CREATE TABLE department_report (
    dept_code TEXT PRIMARY KEY, dept_name TEXT, category TEXT, period TEXT,
    payload TEXT);
""")
conn.executemany(
    "INSERT INTO kpi_definition VALUES (?,?,?,?,?,?,?,?,?,?)",
    [(k, d["name"], d["domain"], d["unit"], d["formula"], d["target"],
      d["target_source"], d["source_system"], d.get("figure_status", "final"),
      d.get("caveat")) for k, d in KPI_DEFINITIONS.items()])
conn.executemany(
    "INSERT INTO department_report VALUES (?,?,?,?,?)",
    [(r["dept_code"], r["dept_name"], r["category"], REPORTING_PERIOD, json.dumps(r))
     for r in (_cast(x) for x in csv.DictReader(io.StringIO(DEPARTMENT_REPORTS_CSV)))])
conn.commit()


class SQLiteBackend:
    name = "sqlite"

    def __init__(self, path: Path):
        self.path = path

    def _rows(self, sql: str, args=()) -> List[sqlite3.Row]:
        c = sqlite3.connect(self.path)
        c.row_factory = sqlite3.Row
        try:
            return c.execute(sql, args).fetchall()
        finally:
            c.close()

    def kpi_definitions(self, kpi_ids: List[str] = None) -> Dict[str, Any]:
        if kpi_ids:
            marks = ",".join("?" * len(kpi_ids))
            rows = self._rows(
                f"SELECT * FROM kpi_definition WHERE kpi_id IN ({marks})", kpi_ids)
            if len(rows) != len(kpi_ids):
                found = {r["kpi_id"] for r in rows}
                raise MCPError(E_NOT_FOUND,
                               f"unknown KPI id(s): {sorted(set(kpi_ids) - found)}")
        else:
            rows = self._rows("SELECT * FROM kpi_definition ORDER BY kpi_id")
        defs = {}
        for r in rows:
            d = dict(KPI_DEFINITIONS[r["kpi_id"]])   # formula fields for the calculator
            d.update({"name": r["name"], "domain": r["domain"], "target": r["target"],
                      "target_source": r["target_source"],
                      "source_system": r["source_system"]})
            defs[r["kpi_id"]] = d
        return {"definitions": defs, "count": len(defs),
                "source_system": "Institutional KPI register (SQLite)"}

    # The SAME resource the file backend serves — used in Cell 6 to swap sources.
    def department_reports(self, period: str, category: str = None) -> Dict[str, Any]:
        sql = "SELECT payload FROM department_report WHERE period = ?"
        args = [period]
        if category:
            sql += " AND category = ?"
            args.append(category)
        rows = [json.loads(r["payload"]) for r in self._rows(sql, args)]
        if not rows:
            raise MCPError(E_NOT_FOUND,
                           f"no rows for period '{period}'"
                           + (f" and category '{category}'" if category else ""))
        return {"rows": rows, "row_count": len(rows), "period": period,
                "source_system": "Department reports (SQLite table)"}


# ---- backend 3: a mock HTTP API -------------------------------------------
class APIBackend:
    name = "http:benchmark-api"
    LATENCY_S = 0.02

    def external_benchmark(self, kpi_id: str) -> Dict[str, Any]:
        time.sleep(self.LATENCY_S)              # a network call is not free
        if kpi_id not in KPI_DEFINITIONS:
            raise MCPError(E_BAD_REQUEST, f"'{kpi_id}' is not a known KPI id")
        record = BENCHMARK_API_DB.get(kpi_id)
        if record is None:
            raise MCPError(E_NOT_FOUND,
                           f"no published external benchmark for {kpi_id} "
                           f"({KPI_DEFINITIONS[kpi_id]['name']})")
        return {"kpi_id": kpi_id, **record,
                "source_system": "External benchmark API (NIRF cohort)"}


file_backend, sqlite_backend, api_backend = (FileBackend(CSV_PATH),
                                             SQLiteBackend(DB_PATH), APIBackend())


# ==========================================================================
# LAB 5 — expose resources, issue scoped tokens
# ==========================================================================

server = MCPServer("edss-connector")

server.expose(Resource(
    name="department_reports",
    description="Per-department metric rows for one reporting period.",
    params={"period": {"type": "string", "required": True},
            "category": {"type": "string", "required": False,
                         "enum": ["circuit", "core"]}},
    returns="{rows, row_count, period, source_system}",
    scope="read:departments", backend=file_backend.name,
    handler=file_backend.department_reports))

server.expose(Resource(
    name="kpi_definitions",
    description="Approved institution KPI definitions: formula, unit, target, authority.",
    params={"kpi_ids": {"type": "array", "required": False}},
    returns="{definitions, count, source_system}",
    scope="read:kpi-register", backend=sqlite_backend.name,
    handler=sqlite_backend.kpi_definitions))

server.expose(Resource(
    name="external_benchmark",
    description="Published peer median for a KPI. Coverage is partial.",
    params={"kpi_id": {"type": "string", "required": True}},
    returns="{kpi_id, peer_median, cohort, year, source_system}",
    scope="read:benchmarks", backend=api_backend.name,
    handler=api_backend.external_benchmark))

# ---- scoped tokens: the Registrar's authorisation, expressed as data -------
server.issue_token("tok-coordinator", "apniLeap AURA Coordinator",
                   ["read:departments", "read:kpi-register", "read:benchmarks"])
server.issue_token("tok-academic-agent", "Academic Agent",
                   ["read:departments", "read:kpi-register"])   # no benchmark access
server.issue_token("tok-intern", "Summer intern (read-only KPI register)",
                   ["read:kpi-register"])

# ==========================================================================
# A ready-made client for the coordinator
# ==========================================================================

coordinator_client = MCPClient(server, "tok-coordinator")


Overwriting edss_stack.py


### Cell 2 · Swarm primitives

One worker per unit, run in parallel, with per-agent retry and backoff, a result envelope that carries failure as data rather than as an exception, and timings for the parallel-vs-serial comparison.

In [3]:
# =============================================================================
# Cell 2  ·  Swarm primitives — parallel, with the failure cases built in
# =============================================================================
# Concept 8: a swarm is many agents doing the SAME job on DIFFERENT units, run
# at once, then merged. Lab 7 already ran four domain agents in one parallel
# step. What Lab 8 adds is everything that happens when parallel work goes
# wrong:
#
#   an agent is slow          -> timeout
#   an agent fails            -> retry with backoff
#   an agent stays failed     -> partial merge with an explicit coverage gap
#   two agents disagree       -> conflict resolution, not double-counting
#
# Here the unit is the DEPARTMENT: eight agents, one per department.

import json, random, textwrap, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Set

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD,
                       DEPARTMENT_REPORTS_CSV, _typed_rows, ToolRouter)
from edss_stack import coordinator_client


@dataclass
class AgentResult:
    """The envelope every swarm agent returns — success or failure."""
    unit: str
    ok: bool
    data: Optional[Dict[str, Any]] = None
    error: Optional[str] = None
    attempts: int = 1
    ms: float = 0.0


class Swarm:
    """Runs one worker per unit, in parallel, with retry and timeout."""

    def __init__(self, worker: Callable[[str], Dict[str, Any]],
                 max_retries: int = 2, backoff_s: float = 0.05,
                 timeout_s: float = 2.0):
        self.worker = worker
        self.max_retries = max_retries
        self.backoff_s = backoff_s
        self.timeout_s = timeout_s
        self.log: List[Dict[str, Any]] = []

    def _run_one(self, unit: str) -> AgentResult:
        t0 = time.perf_counter()
        last = None
        for attempt in range(1, self.max_retries + 2):
            try:
                data = self.worker(unit)
                return AgentResult(unit, True, data, None, attempt,
                                   (time.perf_counter() - t0) * 1000)
            except Exception as exc:
                last = f"{type(exc).__name__}: {exc}"
                self.log.append({"unit": unit, "attempt": attempt, "error": last})
                if attempt <= self.max_retries:
                    time.sleep(self.backoff_s * attempt)      # backoff
        return AgentResult(unit, False, None, last, self.max_retries + 1,
                           (time.perf_counter() - t0) * 1000)

    def run(self, units: List[str]) -> Dict[str, AgentResult]:
        started = time.perf_counter()
        results: Dict[str, AgentResult] = {}
        with ThreadPoolExecutor(max_workers=len(units)) as pool:
            futures = {pool.submit(self._run_one, u): u for u in units}
            for fut in as_completed(futures, timeout=self.timeout_s * 4):
                r = fut.result()
                results[r.unit] = r
        self.wall_ms = (time.perf_counter() - started) * 1000
        self.serial_ms = sum(r.ms for r in results.values())
        return {u: results[u] for u in units if u in results}


print("Swarm primitives ready: one worker per unit, run in parallel, with")
print("per-agent retry + backoff, a partial-result envelope, and timings for")
print("the parallel-vs-serial comparison.")


Swarm primitives ready: one worker per unit, run in parallel, with
per-agent retry + backoff, a partial-result envelope, and timings for
the parallel-vs-serial comparison.


### Cell 3 · Eight agents — and three things that go wrong

The faults are deliberate and named, because a swarm that only ever succeeds demonstrates nothing about merging.

`PUB-J1`–`J3` appear in **both** the CSE and ECE lists — not a bug in the fixture, but what genuinely happens when two departments co-author a paper and each reports it.

In [4]:
# =============================================================================
# Cell 3  ·  Eight department agents — and three things that go wrong
# =============================================================================
# The faults are deliberate and named, because a swarm that only ever succeeds
# demonstrates nothing about merging.
#
#   CHEM  — a flaky source: fails once, succeeds on retry
#   CIVIL — a dead source: never succeeds, so the merge must be partial
#   CSE/ECE — both legitimately claim the same three joint publications, which
#             the merge must count ONCE (Chapter 15 Research guardrail:
#             "never double-count a cross-department collaboration")

ROWS = {r["dept_code"]: r for r in _typed_rows(DEPARTMENT_REPORTS_CSV)}
UNITS = list(ROWS)

# Publication ids per department. PUB-J1..J3 are joint CSE+ECE papers and
# appear in BOTH lists — exactly as they do in the two departments' own returns.
JOINT = ["PUB-J1", "PUB-J2", "PUB-J3"]
PUBLICATION_IDS = {}
for code, row in ROWS.items():
    n = row["publications"]
    if code == "CSE":
        own = [f"PUB-{code}-{i}" for i in range(1, n - len(JOINT) + 1)]
        PUBLICATION_IDS[code] = own + JOINT
    elif code == "ECE":
        own = [f"PUB-{code}-{i}" for i in range(1, n - len(JOINT) + 1)]
        PUBLICATION_IDS[code] = own + JOINT
    else:
        PUBLICATION_IDS[code] = [f"PUB-{code}-{i}" for i in range(1, n + 1)]

FAULTS = {"CHEM": {"fail_until_attempt": 2}, "CIVIL": {"always_fail": True}}
ATTEMPTS = defaultdict(int)
AGENT_LATENCY_S = 0.06          # each agent talks to a departmental system


def department_agent(unit: str) -> Dict[str, Any]:
    """One agent, one department. Same code for all eight."""
    ATTEMPTS[unit] += 1
    time.sleep(AGENT_LATENCY_S)

    fault = FAULTS.get(unit, {})
    if fault.get("always_fail"):
        raise ConnectionError(f"{unit} departmental server unreachable")
    if ATTEMPTS[unit] < fault.get("fail_until_attempt", 0):
        raise TimeoutError(f"{unit} reporting system timed out")

    row = ROWS[unit]
    return {
        "dept_code": unit, "dept_name": row["dept_name"], "category": row["category"],
        "students_appeared": row["students_appeared"],
        "students_passed": row["students_passed"],
        "sanctioned_faculty": row["sanctioned_faculty"],
        "publication_ids": PUBLICATION_IDS[unit],
        "publication_count_claimed": len(PUBLICATION_IDS[unit]),
        "eligible_students": row["eligible_students"],
        "placed_students": row["placed_students"],
        "source_system": f"{unit} departmental reporting system",
    }


print(f"{'dept':<8}{'claims':>8}   {'joint papers also claimed by another dept':<44}fault")
for u in UNITS:
    joint_here = [p for p in PUBLICATION_IDS[u] if p in JOINT]
    fault = ("flaky — fails once, then succeeds" if u == "CHEM"
             else "DEAD — never responds" if u == "CIVIL" else "-")
    print(f"{u:<8}{len(PUBLICATION_IDS[u]):>8}   "
          f"{', '.join(joint_here) if joint_here else '-':<44}{fault}")

naive_total = sum(len(v) for v in PUBLICATION_IDS.values())
true_total = len({p for v in PUBLICATION_IDS.values() for p in v})
print(f"\n  sum of what the eight agents claim : {naive_total}")
print(f"  distinct publications              : {true_total}   "
      f"({naive_total - true_total} joint papers claimed by two departments each)")
print(f"\n  Note what this means for Labs 1-7: the flat department-reports export")
print(f"  carries the per-department claim, so its institution total is {naive_total} —")
print(f"  the double-counted figure. Only a swarm that merges IDENTITIES rather than")
print(f"  COUNTS can see that the true figure is {true_total}.")


dept      claims   joint papers also claimed by another dept   fault
CSE           58   PUB-J1, PUB-J2, PUB-J3                      -
ECE           34   PUB-J1, PUB-J2, PUB-J3                      -
ISE           21   -                                           -
EEE           16   -                                           -
MECH          19   -                                           -
CIVIL         11   -                                           DEAD — never responds
CHEM          14   -                                           flaky — fails once, then succeeds
BT            12   -                                           -

  sum of what the eight agents claim : 185
  distinct publications              : 182   (3 joint papers claimed by two departments each)

  Note what this means for Labs 1-7: the flat department-reports export
  carries the per-department claim, so its institution total is 185 —
  the double-counted figure. Only a swarm that merges IDENTITIES rather than
 

### Cell 4 · Run the swarm

Watch the `attempts` column: CHEM takes 2, CIVIL takes 3 and still fails. The wall time is roughly one agent's latency; the summed time is what running them one after another would have cost.

In [5]:
# =============================================================================
# Cell 4  ·  Run the swarm
# =============================================================================
# Eight agents, one per department, all at once — with two of them failing.

swarm = Swarm(department_agent, max_retries=2, backoff_s=0.05)
results = swarm.run(UNITS)

print("=" * 88)
print("SWARM RUN")
print("=" * 88)
print(f"  {'dept':<8}{'ok':<7}{'attempts':<11}{'ms':>9}   detail")
for u in UNITS:
    r = results[u]
    detail = (f"{r.data['publication_count_claimed']} publications claimed" if r.ok
              else r.error)
    print(f"  {u:<8}{str(r.ok):<7}{r.attempts:<11}{r.ms:>9.1f}   {detail}")

ok_units = [u for u, r in results.items() if r.ok]
failed_units = [u for u, r in results.items() if not r.ok]
retried = [u for u, r in results.items() if r.ok and r.attempts > 1]

print(f"\n  succeeded            : {len(ok_units)}/{len(UNITS)}  {ok_units}")
print(f"  recovered on retry   : {retried}")
print(f"  permanently failed   : {failed_units}")
print(f"\n  wall time (parallel) : {swarm.wall_ms:.0f} ms")
print(f"  sum of agent times   : {swarm.serial_ms:.0f} ms  "
      f"(what running them one after another would have cost)")
print(f"  saved by the swarm   : {swarm.serial_ms - swarm.wall_ms:.0f} ms")
print(f"\n  retry log ({len(swarm.log)} failed attempts):")
for e in swarm.log:
    print(f"      {e['unit']} attempt {e['attempt']}: {e['error']}")


SWARM RUN
  dept    ok     attempts          ms   detail
  CSE     True   1               63.1   58 publications claimed
  ECE     True   1               63.2   34 publications claimed
  ISE     True   1               63.1   21 publications claimed
  EEE     True   1               63.0   16 publications claimed
  MECH    True   1               63.0   19 publications claimed
  CIVIL   False  3              368.9   ConnectionError: CIVIL departmental server unreachable
  CHEM    True   2              187.8   14 publications claimed
  BT      True   1               62.9   12 publications claimed

  succeeded            : 7/8  ['CSE', 'ECE', 'ISE', 'EEE', 'MECH', 'CHEM', 'BT']
  recovered on retry   : ['CHEM']
  permanently failed   : ['CIVIL']

  wall time (parallel) : 373 ms
  sum of agent times   : 935 ms  (what running them one after another would have cost)
  saved by the swarm   : 561 ms

  retry log (4 failed attempts):
      CHEM attempt 1: TimeoutError: CHEM reporting system timed

### Cell 5 · The merge

Two rules make the merge more than a sum:

1. **Counts are merged as identities**, so a paper claimed by two departments is counted once.
2. **Coverage is tracked**, so a partial result knows it is partial.

The KPI effect is small and real — and it is an *overstatement* that the reconciliation removes.

In [6]:
# =============================================================================
# Cell 5  ·  The merge — resolve the conflict, do not double-count
# =============================================================================
# The merge is the part that makes a swarm useful. Two rules:
#
#   counts are merged as IDENTITIES, so a paper claimed by two departments is
#   counted once;
#   coverage is tracked, so a partial result is labelled partial.

def merge(results: Dict[str, AgentResult]) -> Dict[str, Any]:
    reporting = {u: r.data for u, r in results.items() if r.ok}
    missing = [u for u, r in results.items() if not r.ok]

    # --- conflict resolution: union of ids, not sum of counts ---------------
    claimed_by: Dict[str, List[str]] = defaultdict(list)
    for unit, d in reporting.items():
        for pub in d["publication_ids"]:
            claimed_by[pub].append(unit)
    contested = {p: u for p, u in claimed_by.items() if len(u) > 1}

    naive = sum(d["publication_count_claimed"] for d in reporting.values())
    reconciled = len(claimed_by)

    totals = {
        "students_appeared": sum(d["students_appeared"] for d in reporting.values()),
        "students_passed": sum(d["students_passed"] for d in reporting.values()),
        "sanctioned_faculty": sum(d["sanctioned_faculty"] for d in reporting.values()),
        "eligible_students": sum(d["eligible_students"] for d in reporting.values()),
        "placed_students": sum(d["placed_students"] for d in reporting.values()),
    }
    coverage = len(reporting) / len(results) * 100
    return {
        "reporting_departments": sorted(reporting),
        "missing_departments": missing,
        "coverage_pct": round(coverage, 2),
        "complete": not missing,
        "publications_naive_sum": naive,
        "publications_reconciled": reconciled,
        "double_counted_removed": naive - reconciled,
        "contested_publications": {p: sorted(u) for p, u in contested.items()},
        "totals": totals,
        "kpis": {
            "pass_percentage": round(totals["students_passed"]
                                     / totals["students_appeared"] * 100, 4),
            "publications_per_faculty_naive": round(naive / totals["sanctioned_faculty"], 4),
            "publications_per_faculty_reconciled": round(
                reconciled / totals["sanctioned_faculty"], 4),
            "placement_percentage": round(totals["placed_students"]
                                          / totals["eligible_students"] * 100, 4),
        },
    }


merged = merge(results)

print("=" * 88)
print("MERGE")
print("=" * 88)
print(f"  reporting departments : {len(merged['reporting_departments'])}/8  "
      f"{merged['reporting_departments']}")
print(f"  missing               : {merged['missing_departments']}")
print(f"  coverage              : {merged['coverage_pct']}%   "
      f"complete: {merged['complete']}")

print(f"\n  CONFLICT RESOLUTION")
print(f"    publications, summed as counts    : {merged['publications_naive_sum']}")
print(f"    publications, merged as identities: {merged['publications_reconciled']}")
print(f"    double-counted claims removed     : {merged['double_counted_removed']}")
for pub, owners in merged["contested_publications"].items():
    print(f"        {pub} claimed by {owners} — counted once")

n = merged["kpis"]["publications_per_faculty_naive"]
r = merged["kpis"]["publications_per_faculty_reconciled"]
print(f"\n  Effect on the KPI leadership actually sees:")
print(f"    publications per faculty, naive      : {n:.4f}")
print(f"    publications per faculty, reconciled : {r:.4f}")
print(f"    overstatement avoided                : {n - r:+.4f} per faculty")
print(f"\n  Chapter 15, Research Agent guardrail: \"never double-count a")
print(f"  cross-department collaboration\". This is that guardrail, enforced in the")
print(f"  merge rather than trusted to the departments.")


MERGE
  reporting departments : 7/8  ['BT', 'CHEM', 'CSE', 'ECE', 'EEE', 'ISE', 'MECH']
  missing               : ['CIVIL']
  coverage              : 87.5%   complete: False

  CONFLICT RESOLUTION
    publications, summed as counts    : 174
    publications, merged as identities: 171
    double-counted claims removed     : 3
        PUB-J1 claimed by ['CSE', 'ECE'] — counted once
        PUB-J2 claimed by ['CSE', 'ECE'] — counted once
        PUB-J3 claimed by ['CSE', 'ECE'] — counted once

  Effect on the KPI leadership actually sees:
    publications per faculty, naive      : 1.2429
    publications per faculty, reconciled : 1.2214
    overstatement avoided                : +0.0215 per faculty

  Chapter 15, Research Agent guardrail: "never double-count a
  cross-department collaboration". This is that guardrail, enforced in the
  merge rather than trusted to the departments.


### Cell 6 · What a partial merge may and may not say

A swarm that quietly drops a dead unit produces a number that looks institution-wide and is not. So `publish_institution_figure` refuses at 87.5% coverage — and then shows what the merge **is** allowed to say: the same figure, explicitly labelled partial with the missing department named.

Same rule Lab 2 applied to the provisional Finance figure, now applied to coverage.

In [7]:
# =============================================================================
# Cell 6  ·  What a partial merge may and may not say
# =============================================================================
# Chapter 15's Academic guardrail: "flag any department with stale or missing
# data". A swarm that quietly drops a dead unit produces a number that looks
# institution-wide and is not.

class PartialCoverageError(Exception):
    pass


def publish_institution_figure(merged: Dict[str, Any], kpi_label: str,
                               value: float, force: bool = False) -> str:
    if not merged["complete"] and not force:
        raise PartialCoverageError(
            f"refusing to publish '{kpi_label}' as an institution figure: only "
            f"{merged['coverage_pct']}% coverage, missing "
            f"{merged['missing_departments']}")
    return f"{kpi_label} = {value}"


print("=" * 88)
print("1 · A PARTIAL RESULT CANNOT BE PUBLISHED AS AN INSTITUTION FIGURE")
print("=" * 88)
try:
    publish_institution_figure(merged, "Publications per faculty",
                               merged["kpis"]["publications_per_faculty_reconciled"])
    print("  [!!] a 7-of-8 figure was published as institution-wide")
except PartialCoverageError as exc:
    print(f"  [REFUSED] {exc}")

print("\n" + "=" * 88)
print("2 · WHAT IT MAY SAY INSTEAD")
print("=" * 88)
labelled = (f"Publications per faculty (PARTIAL — "
            f"{len(merged['reporting_departments'])} of {len(results)} departments, "
            f"{', '.join(merged['missing_departments'])} not reporting): "
            f"{merged['kpis']['publications_per_faculty_reconciled']:.4f}")
print(f"  {textwrap.fill(labelled, 82, subsequent_indent='  ')}")
print(f"\n  The figure is still useful — it is simply not allowed to masquerade as")
print(f"  complete. That is the same rule Lab 2 applied to the provisional Finance")
print(f"  figure, now applied to coverage instead of reconciliation status.")

print("\n" + "=" * 88)
print("3 · A DATA-QUALITY FINDING FOR THE REGISTRAR")
print("=" * 88)
flat_total = sum(r["publications"] for r in ROWS.values())
distinct_all_eight = len({p for v in PUBLICATION_IDS.values() for p in v})
faculty_all_eight = sum(r["sanctioned_faculty"] for r in ROWS.values())
print(f"  publications in the flat department-reports export : {flat_total}")
print(f"  distinct publication identities across all eight   : {distinct_all_eight}")
print(f"  papers counted twice in the export                 : "
      f"{flat_total - distinct_all_eight}")
print(f"\n  KPI-R1 as Labs 1-7 computed it : "
      f"{flat_total / faculty_all_eight:.4f} per faculty")
print(f"  KPI-R1 with identities merged  : "
      f"{distinct_all_eight / faculty_all_eight:.4f} per faculty")
print(f"\n  The export was never wrong about any single department — each one")
print(f"  correctly reported its own joint papers. The double-count only becomes")
print(f"  visible when the units are merged by identity, which is precisely what a")
print(f"  swarm-plus-merge does and a flat sum cannot.")


1 · A PARTIAL RESULT CANNOT BE PUBLISHED AS AN INSTITUTION FIGURE
  [REFUSED] refusing to publish 'Publications per faculty' as an institution figure: only 87.5% coverage, missing ['CIVIL']

2 · WHAT IT MAY SAY INSTEAD
  Publications per faculty (PARTIAL — 7 of 8 departments, CIVIL not reporting):
  1.2214

  The figure is still useful — it is simply not allowed to masquerade as
  complete. That is the same rule Lab 2 applied to the provisional Finance
  figure, now applied to coverage instead of reconciliation status.

3 · A DATA-QUALITY FINDING FOR THE REGISTRAR
  publications in the flat department-reports export : 185
  distinct publication identities across all eight   : 182
  papers counted twice in the export                 : 3

  KPI-R1 as Labs 1-7 computed it : 1.1859 per faculty
  KPI-R1 with identities merged  : 1.1667 per faculty

  The export was never wrong about any single department — each one
  correctly reported its own joint papers. The double-count only becomes
  v

### Cell 7 · Acceptance checks — the rubric, executed

In [8]:
# =============================================================================
# Cell 7  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 8: parallelise the per-unit work and merge the best results,
# resolving conflicts and surviving partial failure.

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


check("H1  one agent per unit, all run in parallel",
      len(results) == len(UNITS) == 8,
      f"{len(results)} department agents dispatched in one ThreadPoolExecutor")

check("H2  parallel execution is genuinely faster than serial",
      swarm.wall_ms < swarm.serial_ms,
      f"{swarm.wall_ms:.0f} ms wall vs {swarm.serial_ms:.0f} ms summed "
      f"({swarm.serial_ms - swarm.wall_ms:.0f} ms saved)")

check("H3  a transient failure is retried and recovers",
      "CHEM" in retried and results["CHEM"].ok and results["CHEM"].attempts == 2,
      f"CHEM failed once then succeeded on attempt {results['CHEM'].attempts}")

check("H4  a permanent failure is retried, then given up on cleanly",
      not results["CIVIL"].ok and results["CIVIL"].attempts == 3
      and results["CIVIL"].error,
      f"CIVIL: {results['CIVIL'].attempts} attempts, final error recorded")

check("H5  one dead unit does not abort the whole swarm",
      len(ok_units) == 7 and merged["reporting_departments"],
      f"{len(ok_units)} of 8 agents still produced a usable result")

check("H6  every failed attempt is logged",
      len(swarm.log) == 4,
      f"{len(swarm.log)} failed attempts recorded (1 CHEM + 3 CIVIL)")

check("H7  the merge detects the cross-department conflict",
      len(merged["contested_publications"]) == 3
      and all(sorted(v) == ["CSE", "ECE"]
              for v in merged["contested_publications"].values()),
      f"{len(merged['contested_publications'])} papers claimed by two departments each")

check("H8  contested items are counted once, not twice",
      merged["publications_reconciled"]
      == merged["publications_naive_sum"] - merged["double_counted_removed"],
      f"{merged['publications_naive_sum']} claimed -> "
      f"{merged['publications_reconciled']} distinct")

check("H9  the conflict actually changes the KPI leadership sees",
      merged["kpis"]["publications_per_faculty_naive"]
      > merged["kpis"]["publications_per_faculty_reconciled"],
      f"{merged['kpis']['publications_per_faculty_naive']:.4f} -> "
      f"{merged['kpis']['publications_per_faculty_reconciled']:.4f} per faculty")

check("H10 coverage is tracked and the result is marked incomplete",
      merged["coverage_pct"] == 87.5 and merged["complete"] is False,
      f"coverage {merged['coverage_pct']}%, missing {merged['missing_departments']}")

check("H11 a partial merge cannot be published as an institution figure",
      not merged["complete"],
      "publish_institution_figure raised PartialCoverageError at 87.5% coverage")

check("H12 the flat export's double-count is only visible after an identity merge",
      flat_total - distinct_all_eight == 3,
      f"export says {flat_total}; distinct identities across all eight are "
      f"{distinct_all_eight}")

print("=" * 88)
print("LAB 8 ACCEPTANCE CHECKS")
print("=" * 88)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 88)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 8 acceptance criteria satisfied.")


LAB 8 ACCEPTANCE CHECKS
  [PASS]  H1  one agent per unit, all run in parallel
          8 department agents dispatched in one ThreadPoolExecutor
  [PASS]  H2  parallel execution is genuinely faster than serial
          373 ms wall vs 935 ms summed (561 ms saved)
  [PASS]  H3  a transient failure is retried and recovers
          CHEM failed once then succeeded on attempt 2
  [PASS]  H4  a permanent failure is retried, then given up on cleanly
          CIVIL: 3 attempts, final error recorded
  [PASS]  H5  one dead unit does not abort the whole swarm
          7 of 8 agents still produced a usable result
  [PASS]  H6  every failed attempt is logged
          4 failed attempts recorded (1 CHEM + 3 CIVIL)
  [PASS]  H7  the merge detects the cross-department conflict
          3 papers claimed by two departments each
  [PASS]  H8  contested items are counted once, not twice
          174 claimed -> 171 distinct
  [PASS]  H9  the conflict actually changes the KPI leadership sees
          

### Cell 8 · Deliverables

In [9]:
# =============================================================================
# Cell 8  ·  Deliverables
# =============================================================================
outdir = Path("lab8_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab8_swarm_results.json": {u: {"ok": r.ok, "attempts": r.attempts,
                                    "ms": round(r.ms, 2), "error": r.error,
                                    "claimed": (r.data or {}).get(
                                        "publication_count_claimed")}
                                for u, r in results.items()},
    "lab8_retry_log.json": swarm.log,
    "lab8_merged_result.json": merged,
    "lab8_timings.json": {"parallel_wall_ms": round(swarm.wall_ms, 2),
                          "serial_sum_ms": round(swarm.serial_ms, 2),
                          "saved_ms": round(swarm.serial_ms - swarm.wall_ms, 2),
                          "agents": len(UNITS)},
    "lab8_data_quality_finding.json": {
        "flat_export_publications": flat_total,
        "distinct_identities_all_eight": distinct_all_eight,
        "double_counted": flat_total - distinct_all_eight,
        "kpi_r1_as_computed_in_labs_1_7": round(flat_total / faculty_all_eight, 4),
        "kpi_r1_identity_merged": round(distinct_all_eight / faculty_all_eight, 4),
        "contested": {p: sorted(u) for p, u in merged["contested_publications"].items()}},
    "lab8_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(json.dumps(obj, indent=2, default=str))

print("=" * 88)
print("ARTIFACTS WRITTEN")
print("=" * 88)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")
print(f"\n  Agents dispatched  : {len(UNITS)}  ({len(ok_units)} ok, "
      f"{len(failed_units)} failed, {len(retried)} recovered on retry)")
print(f"  Parallel speed-up  : {swarm.serial_ms - swarm.wall_ms:.0f} ms")
print(f"  Conflicts resolved : {merged['double_counted_removed']}")
print(f"  Coverage           : {merged['coverage_pct']}%")
print(f"  Acceptance checks  : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab8_artifacts/lab8_swarm_results.json  (901 bytes)
  lab8_artifacts/lab8_retry_log.json  (464 bytes)
  lab8_artifacts/lab8_merged_result.json  (825 bytes)
  lab8_artifacts/lab8_timings.json  (96 bytes)
  lab8_artifacts/lab8_data_quality_finding.json  (345 bytes)
  lab8_artifacts/lab8_acceptance_checks.json  (1,952 bytes)

  Agents dispatched  : 8  (7 ok, 1 failed, 1 recovered on retry)
  Parallel speed-up  : 561 ms
  Conflicts resolved : 3
  Coverage           : 87.5%
  Acceptance checks  : 12/12 passed


---

## Where Labs 1–8 leave the worklet

**Phase A · Foundations (Labs 1–6) and Phase B · Orchestration (Labs 7–8) are complete.**

| Lab | What the worklet gained |
|---|---|
| 1 | A coordinator loop that clarifies, plans and **revises by patching** |
| 2 | A typed tool layer — every figure computed, none asserted |
| 3 | Reusable, schema-typed **skills** with declared tool surfaces |
| 4 | **Memory**: run state vs an append-only archive, tag + similarity retrieval |
| 5 | One **MCP-style connector** over file + SQLite + API, with scopes and audit |
| 6 | A **runtime**: budgets, gates, audit log, checkpoint and resume |
| 7 | An explicit **node graph** with parallel branches and a validation back-edge |
| 8 | A **fault-tolerant swarm** with conflict resolution and coverage tracking |

The Streamlit dashboard in `edss_dashboard/` runs this system end to end.

**What Phase C and D add (Labs 9–14):** evidence-grounded deep research, cloud operations, the agentic SDLC, safety and governance with rollback, discipline-specific agents, and the integrated capstone.